# EX_07 — Reranking y optimización (ejercicios)

**Notebook de referencia:** `notebook/07_Reranking_Optimizacion.ipynb`

**Tiempo orientativo:** ~30 minutos.


## Actividad 1 — Reordenar por cross-score simulado

Dada una query y 5 documentos, supón que tienes scores de un bi-encoder (baratos) y scores de un cross-encoder (caros). Implementa: tomar top-4 por bi-encoder y reordenar solo esos 4 por cross-score.


In [1]:
import numpy as np

query = "latency vs throughput"
docs = [f"doc{i}: ..." for i in range(5)]

# Scores ficticios dados por la imagen
bi_scores = np.array([0.72, 0.81, 0.55, 0.78, 0.60])
cross_scores = np.array([0.1, 0.9, 0.2, 0.85, 0.3]) # Alineados con docs original

# --- TODO: two-stage ranking ---

# 1. Obtener los índices de los top-4 clasificados por el bi-encoder
# argsort devuelve de menor a mayor, por eso usamos [-4:] y luego invertimos [::-1]
top4_bi_indices = np.argsort(bi_scores)[-4:][::-1]

print("Top-4 por Bi-Encoder (índices):", top4_bi_indices)

# 2. Extraer los cross-scores correspondientes a esos 4 documentos seleccionados
selected_cross_scores = cross_scores[top4_bi_indices]

# 3. Reordenar ese top-4 basándonos exclusivamente en sus cross_scores
# Obtenemos las posiciones relativas dentro del subconjunto ordenadas de mayor a menor cross-score
rerank_sub_indices = np.argsort(selected_cross_scores)[::-1]

# 4. Mapear los índices de vuelta a la lista original de documentos
final_indices = top4_bi_indices[rerank_sub_indices]

print("\n--- ORDEN FINAL REORDENADO POR CROSS-ENCODER ---")
for pos, idx in enumerate(final_indices, 1):
    print(f"Puesto {pos}: {docs[idx]} (Bi-Score: {bi_scores[idx]:.2f}, Cross-Score: {cross_scores[idx]:.2f})")


Top-4 por Bi-Encoder (índices): [1 3 0 4]

--- ORDEN FINAL REORDENADO POR CROSS-ENCODER ---
Puesto 1: doc1: ... (Bi-Score: 0.81, Cross-Score: 0.90)
Puesto 2: doc3: ... (Bi-Score: 0.78, Cross-Score: 0.85)
Puesto 3: doc4: ... (Bi-Score: 0.60, Cross-Score: 0.30)
Puesto 4: doc0: ... (Bi-Score: 0.72, Cross-Score: 0.10)


## Actividad 2 — MMR esquemático

En pseudocódigo en Python (sin librería), bosqueja 5 líneas de selección **MMR** (balance relevancia / diversidad).


In [2]:
# TODO: MMR pseudocode as comments or stub function

def calculate_mmr_selection(query_emb, candidate_embs, lambda_param=0.5, top_k=3):
    selected_indices = []
    remaining_indices = list(range(len(candidate_embs)))
    
    # 5 líneas clave de la lógica interna de selección iterativa MMR:
    for _ in range(top_k):
        # 1. Calcular similitud de todos los candidatos restantes con la Query
        sim_query = [cosine_similarity(candidate_embs[i], query_emb) for i in remaining_indices]
        
        # 2. Calcular la máxima similitud de cada candidato con lo que ya hemos seleccionado antes
        sim_selected = [max([cosine_similarity(candidate_embs[i], candidate_embs[s]) for s in selected_indices]) if selected_indices else 0 for i in remaining_indices]
        
        # 3. Aplicar la ecuación MMR matemática (Balance de beneficio marginal vs redundancia)
        mmr_scores = [lambda_param * sq - (1 - lambda_param) * ss for sq, ss in zip(sim_query, sim_selected)]
        
        # 4. Escoger el índice candidato que ha maximizado la puntuación MMR en esta vuelta
        best_candidate_idx = remaining_indices[np.argmax(mmr_scores)]
        
        # 5. Mover el elegido desde la lista de disponibles a la lista definitiva de seleccionados
        selected_indices.append(best_candidate_idx)
        remaining_indices.remove(best_candidate_idx)
        
    return selected_indices


## Actividad 3 — Latencia

Estima en markdown (tabla breve) coste relativo: embedding único de query, k llamadas cross-encoder, generación LLM 200 tokens.


| Etapa | Coste relativo (Escala orientativa en ms) | Explicación técnica
| Embedding único de query | ~10 ms - 20 ms| Muy rápido. Es una única pasada (forward) a través de un codificador pequeño (Bi-encoder)|
| k llamadas a Cross-Encoder | 50 ms - 150 ms| Moderado. El Cross-Encoder procesa la query concatenada con cada uno de los $k$ candidatos, lo que exige más cómputo que un embedding simple|
|Generación LLM 200 tokens |2000 ms - 4000 ms|Crítico / El más alto. Es un proceso autorregresivo secuencial; el modelo debe realizar 200 pasadas completas por la red neuronal para predecir cada palabra nueva|


NO ME SALE LA TABLA
